## **SNAP Jupyter demo notebook**
**Geocoded Single Look Complex (GSLC) and GSLC-based InSAR for Sentinel-1...**

In summary, this workflow contains:

- Explain what a GSLC product is and how `GSLC-Terrain-Correction` differs from the classical detected `Terrain-Correction`
- Build a single GSLC product from a Sentinel-1 Stripmap SLC and inspect it
- Run a Stripmap GSLC InSAR pair (interferogram + coherence) using `CreateStack` + `Interferogram`
- Repeat the InSAR pair for Sentinel-1 IW (TOPS) using `TOPSAR-Split` ahead of `GSLC-Terrain-Correction`
- Compare the classical and GSLC InSAR processing chains side-by-side

Complexity: advanced

##### ***About the test data:***

Input Sentinel-1 SLC products are **not** bundled in this repository — they are several GB each. Download an interferometric pair of your choice from the [Copernicus Browser](https://dataspace.copernicus.eu/browser/) (registration required):

- **Stripmap demo**: any two `S1*_SM_SLC__1S*` products acquired over the same area, on the same orbit/track, with a short temporal baseline (e.g. 6/12 days).
- **TOPS demo**: any two `S1*_IW_SLC__1S*` products acquired over the same area, on the same orbit/track, with a short temporal baseline (e.g. 6/12 days).

Place the unzipped `.SAFE` directories (or the `manifest.safe` paths) into the `data/` folder next to this notebook.

Precise Orbit Ephemerides (POE) for each acquisition will be auto-downloaded by SNAP on first use, so an internet connection is needed for the first run.

##### ***Some information on the Python environment:***

In [ ]:
import os
import sys
print("Python version: " + sys.version)

import sysconfig
print("Location of esa_snappy package: " + sysconfig.get_paths()['purelib'] + os.sep + "esa_snappy")

##### ***Import Python packages...***

**Note:** The imports of *esa_snappy* and *snapista* may emit Info/Warning messages from SNAP core. They can be ignored here.

In [ ]:
import esa_snappy
from esa_snappy import ProductIO

import snapista
from snapista import Graph
from snapista import Operator

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

##### ***Convenience plot functions:***

We reuse these throughout the notebook to display real, imaginary, intensity, phase and coherence bands.

In [ ]:
def plot_complex(product, i_band_name, q_band_name, title_prefix=""):
    """Read an I/Q band pair and plot intensity (log) and wrapped phase side by side."""
    i_band = product.getBand(i_band_name)
    q_band = product.getBand(q_band_name)
    w, h = i_band.getRasterWidth(), i_band.getRasterHeight()
    i_data = np.zeros(w * h, np.float32); i_band.readPixels(0, 0, w, h, i_data); i_data.shape = h, w
    q_data = np.zeros(w * h, np.float32); q_band.readPixels(0, 0, w, h, q_data); q_data.shape = h, w
    intensity = i_data * i_data + q_data * q_data
    phase = np.arctan2(q_data, i_data)
    fig, axs = plt.subplots(1, 2, figsize=(12, 5))
    axs[0].set_title(f"{title_prefix} intensity (log)")
    axs[0].imshow(np.log1p(intensity), cmap='gray')
    axs[1].set_title(f"{title_prefix} wrapped phase")
    axs[1].imshow(phase, cmap='hsv', vmin=-np.pi, vmax=np.pi)
    plt.show()

def plot_interferogram(product, phase_band_name, coh_band_name=None):
    """Plot the wrapped interferogram phase and, optionally, the coherence."""
    phi_band = product.getBand(phase_band_name)
    w, h = phi_band.getRasterWidth(), phi_band.getRasterHeight()
    phi = np.zeros(w * h, np.float32); phi_band.readPixels(0, 0, w, h, phi); phi.shape = h, w
    if coh_band_name is None:
        fig, ax = plt.subplots(figsize=(7, 6))
        im = ax.imshow(phi, cmap='hsv', vmin=-np.pi, vmax=np.pi)
        ax.set_title('Wrapped interferogram phase [rad]')
        fig.colorbar(im, ax=ax)
    else:
        coh_band = product.getBand(coh_band_name)
        coh = np.zeros(w * h, np.float32); coh_band.readPixels(0, 0, w, h, coh); coh.shape = h, w
        fig, axs = plt.subplots(1, 2, figsize=(13, 5))
        im0 = axs[0].imshow(phi, cmap='hsv', vmin=-np.pi, vmax=np.pi)
        axs[0].set_title('Wrapped interferogram phase [rad]')
        fig.colorbar(im0, ax=axs[0])
        im1 = axs[1].imshow(coh, cmap='viridis', vmin=0, vmax=1)
        axs[1].set_title('Coherence [0–1]')
        fig.colorbar(im1, ax=axs[1])
    plt.show()

---

### ***What is a GSLC product?***

A **Geocoded Single Look Complex (GSLC)** product is a SAR product where:

1. The pixels are **already in a map projection** (e.g. WGS84 lat/lon) — same as a classical terrain-corrected product.
2. The pixels are **still complex** (I + jQ) — same as a slant-range SLC.

In short, *"a terrain-corrected SLC"*. Compared to a slant-range SLC, the major advantages are:

- Coregistration of two GSLC products reduces to a simple map-coordinate stack — no `Back-Geocoding`, no `Enhanced-Spectral-Diversity` iteration.
- The interferogram phase no longer contains the flat-earth or topographic phase contributions: those have already been removed by `GSLC-Terrain-Correction`. Downstream operators key on the abstract-metadata flag `is_terrain_corrected = 1` and skip flat-earth / topo-phase subtraction automatically.
- The output is directly usable by GIS-style tools that expect map-projected raster input.

GSLC processing was introduced for missions like NISAR. SNAP supports it across Sentinel-1 (Stripmap and IW), Capella, ICEYE and other SAR sources via the `GSLC-Terrain-Correction` operator.

---

### ***What `GSLC-Terrain-Correction` does***

Naïvely resampling complex I/Q data onto a map grid destroys the phase: a sub-pixel offset between source and target rotates the phasor by an arbitrary amount. `GSLC-Terrain-Correction` solves this in three steps:

1. **Phase flattening** — subtract the carrier (range/azimuth) phase contribution from the input complex pixels, so the remaining signal is locally smooth and safe to interpolate.
2. **High-fidelity complex resampling** — interpolate I and Q separately with a windowed-sinc kernel (`BISINC_5_POINT` by default).
3. **Optional phase restoration** — re-apply the flattened carrier on the geocoded grid so the output is a "true" map-projected SLC. With `outputFlattened=True` (default) the carrier is *kept* removed, which is the form most useful for InSAR.

Important user-facing parameters:

| Parameter | Default | Notes |
|:----------|:--------|:------|
| `demName` | `Copernicus 30m Global DEM` | Auto-downloaded by SNAP on first use |
| `demResamplingMethod` | `BILINEAR_INTERPOLATION` | For sampling the DEM |
| `imgResamplingMethod` | `BISINC_5_POINT_INTERPOLATION` | Complex image resampler — use sinc-family for phase preservation |
| `outputFlattened` | `true` | Keep carrier removed; recommended for InSAR |
| `nodataValueAtSea` | `true` | Mask water bodies (much faster) |
| `saveIncidenceAngleFromEllipsoid` / `saveLocalIncidenceAngle` / `saveLayoverShadowMask` | `false` | Optional auxiliary bands |

For Sentinel-1 IW TOPS data, **always** run `TOPSAR-Split` first to pick a single subswath and a contiguous burst range. Debursted TOPS SLCs are explicitly rejected by `GSLC-Terrain-Correction`.

---

### ***Configure input paths***

Edit the paths below to point to your downloaded SLCs.

In [ ]:
data_dir = os.path.join(os.getcwd(), 'data')
graphs_dir = os.path.join(os.getcwd(), 'graphs')
results_dir = os.path.join(os.getcwd(), 'results')
os.makedirs(graphs_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

# Stripmap pair — adjust to your downloaded products
sm_reference_slc = os.path.join(data_dir, 'S1A_SM_SLC__1SDV_<reference>.SAFE', 'manifest.safe')
sm_secondary_slc = os.path.join(data_dir, 'S1A_SM_SLC__1SDV_<secondary>.SAFE', 'manifest.safe')

# TOPS (IW) pair — adjust to your downloaded products
iw_reference_slc = os.path.join(data_dir, 'S1A_IW_SLC__1SDV_<reference>.SAFE', 'manifest.safe')
iw_secondary_slc = os.path.join(data_dir, 'S1A_IW_SLC__1SDV_<secondary>.SAFE', 'manifest.safe')

---
## ***Part 1 — Stripmap GSLC***
---

### ***1A. Build a single Stripmap GSLC product***

For a Stripmap SLC the chain is short: `Read → Apply-Orbit-File → GSLC-Terrain-Correction → Write`. `Apply-Orbit-File` pulls Precise Orbit Ephemerides (POE) from ESA, which gives the geocoding the position accuracy InSAR needs.

In [ ]:
g_sm_single = Graph()
g_sm_single.add_node(operator=Operator('Read', file=sm_reference_slc), node_id='Read')

g_sm_single.add_node(operator=Operator('Apply-Orbit-File',
                                       orbitType='Sentinel Precise (Auto Download)',
                                       continueOnFail='false'),
                     node_id='ApplyOrbit', source='Read')

g_sm_single.add_node(operator=Operator('GSLC-Terrain-Correction',
                                       demName='Copernicus 30m Global DEM',
                                       imgResamplingMethod='BISINC_5_POINT_INTERPOLATION',
                                       outputFlattened='true'),
                     node_id='GSLC', source='ApplyOrbit')

sm_single_out = os.path.join(results_dir, 'snap_nb_gslc_sm_single.dim')
g_sm_single.add_node(operator=Operator('Write', file=sm_single_out, formatName='BEAM-DIMAP'),
                     node_id='Write', source='GSLC')

g_sm_single.view()
g_sm_single.save_graph(os.path.join(graphs_dir, 'snap_nb_gslc_sm_single.xml'))

##### ***Run the graph and inspect the result:***

In [ ]:
g_sm_single.run()

In [ ]:
p_sm_gslc = ProductIO.readProduct(sm_single_out)
print('Bands:', [b.getName() for b in p_sm_gslc.getBands()])
print('Width x Height:', p_sm_gslc.getSceneRasterWidth(), 'x', p_sm_gslc.getSceneRasterHeight())
print('GeoCoding:', p_sm_gslc.getSceneGeoCoding().__class__.__name__)

# Detect the polarisation suffix actually present
i_name = next(b.getName() for b in p_sm_gslc.getBands()
              if b.getName().startswith('i_') and b.getUnit() == 'real')
q_name = i_name.replace('i_', 'q_', 1)
plot_complex(p_sm_gslc, i_name, q_name, title_prefix='GSLC Stripmap')
p_sm_gslc.dispose()

---

### ***1B. Stripmap GSLC InSAR pair***

We now process both reference and secondary Stripmap SLCs through the same chain, stack them with `CreateStack`, and produce an interferogram. Because both inputs carry `is_terrain_corrected = 1`, `Interferogram` automatically takes the **GSLC code path** and skips flat-earth and topo-phase removal — the operator's `initializeGSLC()` branch detects this flag and switches to a direct complex-conjugate multiply.

In [ ]:
g_sm_insar = Graph()

# --- Reference chain ---
g_sm_insar.add_node(operator=Operator('Read', file=sm_reference_slc), node_id='Read_ref')
g_sm_insar.add_node(operator=Operator('Apply-Orbit-File',
                                      orbitType='Sentinel Precise (Auto Download)'),
                    node_id='Orbit_ref', source='Read_ref')
g_sm_insar.add_node(operator=Operator('GSLC-Terrain-Correction',
                                      demName='Copernicus 30m Global DEM',
                                      imgResamplingMethod='BISINC_5_POINT_INTERPOLATION',
                                      outputFlattened='true'),
                    node_id='GSLC_ref', source='Orbit_ref')

# --- Secondary chain ---
g_sm_insar.add_node(operator=Operator('Read', file=sm_secondary_slc), node_id='Read_sec')
g_sm_insar.add_node(operator=Operator('Apply-Orbit-File',
                                      orbitType='Sentinel Precise (Auto Download)'),
                    node_id='Orbit_sec', source='Read_sec')
g_sm_insar.add_node(operator=Operator('GSLC-Terrain-Correction',
                                      demName='Copernicus 30m Global DEM',
                                      imgResamplingMethod='BISINC_5_POINT_INTERPOLATION',
                                      outputFlattened='true'),
                    node_id='GSLC_sec', source='Orbit_sec')

# --- Stack ---
# Reference and secondary GSLCs share the same map grid, so CreateStack is essentially
# a band-merge here — no Back-Geocoding or ESD iteration needed.
g_sm_insar.add_node(operator=Operator('CreateStack',
                                      extent='Master',
                                      initialOffsetMethod='Product Geolocation'),
                    node_id='Stack', source=['GSLC_ref', 'GSLC_sec'])

# --- Interferogram ---
# Detects is_terrain_corrected=1 and switches to the GSLC code path automatically:
# no flat-earth or topo-phase subtraction.
g_sm_insar.add_node(operator=Operator('Interferogram',
                                      includeCoherence='true',
                                      cohWinAz='10',
                                      cohWinRg='10'),
                    node_id='Ifg', source='Stack')

# Optional Goldstein filter for visual clarity
g_sm_insar.add_node(operator=Operator('GoldsteinPhaseFiltering',
                                      alpha='1.0'),
                    node_id='Goldstein', source='Ifg')

sm_insar_out = os.path.join(results_dir, 'snap_nb_gslc_sm_insar.dim')
g_sm_insar.add_node(operator=Operator('Write', file=sm_insar_out, formatName='BEAM-DIMAP'),
                    node_id='Write', source='Goldstein')

g_sm_insar.save_graph(os.path.join(graphs_dir, 'snap_nb_gslc_sm_insar.xml'))
g_sm_insar.run()

In [ ]:
p_sm_ifg = ProductIO.readProduct(sm_insar_out)
phase_band = next(b.getName() for b in p_sm_ifg.getBands()
                  if b.getName().lower().startswith('phase_'))
coh_band = next((b.getName() for b in p_sm_ifg.getBands()
                 if b.getName().lower().startswith('coh_')), None)
plot_interferogram(p_sm_ifg, phase_band, coh_band)
p_sm_ifg.dispose()

---
## ***Part 2 — TOPS (Sentinel-1 IW) GSLC***
---

### ***2A. Build a single IW GSLC product***

The IW chain inserts `TOPSAR-Split` before `GSLC-Terrain-Correction`: pick one subswath (`IW1` here) and the burst range you want to process. `GSLC-Terrain-Correction` rejects already-debursted TOPS products explicitly, so the burst structure must be preserved.

> Processing chain: `Read → Apply-Orbit-File → TOPSAR-Split → GSLC-Terrain-Correction → Write`

In [ ]:
g_iw_single = Graph()
g_iw_single.add_node(operator=Operator('Read', file=iw_reference_slc), node_id='Read')
g_iw_single.add_node(operator=Operator('Apply-Orbit-File',
                                       orbitType='Sentinel Precise (Auto Download)'),
                     node_id='ApplyOrbit', source='Read')
g_iw_single.add_node(operator=Operator('TOPSAR-Split',
                                       subswath='IW1',
                                       selectedPolarisations='VV',
                                       firstBurstIndex='1',
                                       lastBurstIndex='3'),
                     node_id='Split', source='ApplyOrbit')
g_iw_single.add_node(operator=Operator('GSLC-Terrain-Correction',
                                       demName='Copernicus 30m Global DEM',
                                       imgResamplingMethod='BISINC_5_POINT_INTERPOLATION',
                                       outputFlattened='true'),
                     node_id='GSLC', source='Split')

iw_single_out = os.path.join(results_dir, 'snap_nb_gslc_iw_single.dim')
g_iw_single.add_node(operator=Operator('Write', file=iw_single_out, formatName='BEAM-DIMAP'),
                     node_id='Write', source='GSLC')

g_iw_single.view()
g_iw_single.save_graph(os.path.join(graphs_dir, 'snap_nb_gslc_iw_single.xml'))
g_iw_single.run()

In [ ]:
p_iw_gslc = ProductIO.readProduct(iw_single_out)
print('Bands:', [b.getName() for b in p_iw_gslc.getBands()])
i_name = next(b.getName() for b in p_iw_gslc.getBands()
              if b.getName().startswith('i_') and b.getUnit() == 'real')
q_name = i_name.replace('i_', 'q_', 1)
plot_complex(p_iw_gslc, i_name, q_name, title_prefix='GSLC IW1')
p_iw_gslc.dispose()

---

### ***2B. IW (TOPS) GSLC InSAR pair***

Same shape as the Stripmap InSAR graph, but with `TOPSAR-Split` on each input. Note how short this chain is compared to the classical IW InSAR workflow shown in the comparison table at the end of this notebook.

In [ ]:
g_iw_insar = Graph()

# --- Reference chain ---
g_iw_insar.add_node(operator=Operator('Read', file=iw_reference_slc), node_id='Read_ref')
g_iw_insar.add_node(operator=Operator('Apply-Orbit-File',
                                      orbitType='Sentinel Precise (Auto Download)'),
                    node_id='Orbit_ref', source='Read_ref')
g_iw_insar.add_node(operator=Operator('TOPSAR-Split',
                                      subswath='IW1',
                                      selectedPolarisations='VV',
                                      firstBurstIndex='1',
                                      lastBurstIndex='3'),
                    node_id='Split_ref', source='Orbit_ref')
g_iw_insar.add_node(operator=Operator('GSLC-Terrain-Correction',
                                      demName='Copernicus 30m Global DEM',
                                      imgResamplingMethod='BISINC_5_POINT_INTERPOLATION',
                                      outputFlattened='true'),
                    node_id='GSLC_ref', source='Split_ref')

# --- Secondary chain (same subswath / burst range as reference) ---
g_iw_insar.add_node(operator=Operator('Read', file=iw_secondary_slc), node_id='Read_sec')
g_iw_insar.add_node(operator=Operator('Apply-Orbit-File',
                                      orbitType='Sentinel Precise (Auto Download)'),
                    node_id='Orbit_sec', source='Read_sec')
g_iw_insar.add_node(operator=Operator('TOPSAR-Split',
                                      subswath='IW1',
                                      selectedPolarisations='VV',
                                      firstBurstIndex='1',
                                      lastBurstIndex='3'),
                    node_id='Split_sec', source='Orbit_sec')
g_iw_insar.add_node(operator=Operator('GSLC-Terrain-Correction',
                                      demName='Copernicus 30m Global DEM',
                                      imgResamplingMethod='BISINC_5_POINT_INTERPOLATION',
                                      outputFlattened='true'),
                    node_id='GSLC_sec', source='Split_sec')

# --- Stack & interferogram (same code path as Stripmap; GSLC flag auto-detected) ---
g_iw_insar.add_node(operator=Operator('CreateStack',
                                      extent='Master',
                                      initialOffsetMethod='Product Geolocation'),
                    node_id='Stack', source=['GSLC_ref', 'GSLC_sec'])
g_iw_insar.add_node(operator=Operator('Interferogram',
                                      includeCoherence='true',
                                      cohWinAz='10',
                                      cohWinRg='10'),
                    node_id='Ifg', source='Stack')
g_iw_insar.add_node(operator=Operator('GoldsteinPhaseFiltering',
                                      alpha='1.0'),
                    node_id='Goldstein', source='Ifg')

iw_insar_out = os.path.join(results_dir, 'snap_nb_gslc_iw_insar.dim')
g_iw_insar.add_node(operator=Operator('Write', file=iw_insar_out, formatName='BEAM-DIMAP'),
                    node_id='Write', source='Goldstein')

g_iw_insar.save_graph(os.path.join(graphs_dir, 'snap_nb_gslc_iw_insar.xml'))
g_iw_insar.run()

In [ ]:
p_iw_ifg = ProductIO.readProduct(iw_insar_out)
phase_band = next(b.getName() for b in p_iw_ifg.getBands()
                  if b.getName().lower().startswith('phase_'))
coh_band = next((b.getName() for b in p_iw_ifg.getBands()
                 if b.getName().lower().startswith('coh_')), None)
plot_interferogram(p_iw_ifg, phase_band, coh_band)
p_iw_ifg.dispose()

---

### ***Comparison: classical InSAR vs. GSLC InSAR (Sentinel-1 IW)***

| Step | Classical IW InSAR | GSLC IW InSAR |
|:-----|:-------------------|:--------------|
| 1 | `Read` (ref + sec) | `Read` (ref + sec) |
| 2 | `Apply-Orbit-File` | `Apply-Orbit-File` |
| 3 | `TOPSAR-Split` | `TOPSAR-Split` |
| 4 | `Back-Geocoding` (uses DEM to predict pixel offsets) | `GSLC-Terrain-Correction` (uses DEM + complex resample) |
| 5 | `Enhanced-Spectral-Diversity` (iterative refinement) | `CreateStack` |
| 6 | `Interferogram` (with flat-earth + topo subtraction) | `Interferogram` (auto-skips both, sees `is_terrain_corrected=1`) |
| 7 | `TOPSAR-Deburst` | *(not needed — bursts merged during geocoding)* |
| 8 | `TopoPhaseRemoval` | *(already removed)* |
| 9 | `GoldsteinPhaseFiltering` | `GoldsteinPhaseFiltering` |
| 10 | `Terrain-Correction` (range-Doppler) | *(already geocoded)* |

The GSLC chain trades one heavier per-product step (the complex terrain correction) for the elimination of `Back-Geocoding`, `Enhanced-Spectral-Diversity`, `TOPSAR-Deburst`, `TopoPhaseRemoval` **and** the final `Terrain-Correction`. The result is also map-projected from the start, so it integrates cleanly with downstream GIS tooling.

---

### ***Summary***

What have we learnt in this notebook?

- A **GSLC** product is a complex SAR product that has already been map-projected with phase preservation.
- `GSLC-Terrain-Correction` accomplishes this via phase flattening, sinc-family complex resampling, and (optional) phase restoration. The default `Copernicus 30m Global DEM` is auto-downloaded by SNAP and the default sinc-5 resampler is appropriate for InSAR.
- For Sentinel-1 Stripmap, the GSLC chain is `Read → Apply-Orbit-File → GSLC-Terrain-Correction`.
- For Sentinel-1 IW (TOPS), insert `TOPSAR-Split` before `GSLC-Terrain-Correction`; debursted TOPS products are rejected by design.
- An InSAR pair from two GSLCs is just `CreateStack → Interferogram`: the operator detects `is_terrain_corrected=1` and skips flat-earth/topo phase subtraction.
- Compared to the classical IW InSAR chain, the GSLC chain removes four operators end-to-end and produces a map-projected interferogram directly.